[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/traderbrasil/livro-dinheiro/blob/v2.0/notebooks/cap24_dividas.ipynb)

# Capítulo 24: Saindo das dívidas

Laboratório do livro *Dinheiro: Querer, poder e gastar* (2ª edição), de Flávio Lemos, CMT, CFP®.

Avalanche ou bola de neve? Você simula a quitação de várias dívidas mês a mês, com o exemplo do livro ou com as suas.

**Como usar:** clique em cada célula e aperte o botão ▶ (ou Shift + Enter), de cima para baixo. Na primeira vez, o Colab pede para você entrar com uma conta do Google. Os campos marcados com `# MUDE AQUI` podem ser alterados com os seus números.

*Conteúdo educacional. Não é recomendação de investimento. Resultados passados não garantem resultados futuros. Os dados são baixados no dia em que você roda o notebook; por isso, os seus números podem ser diferentes dos do livro, que foram conferidos em outubro de 2026.*

In [ ]:
# Funções de apoio (não precisa mexer)
import io, time, math, datetime as dt
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plt.rcParams.update({"figure.figsize": (10, 5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
CORES = {"azul": "#1F3864", "verde": "#2E7D32", "laranja": "#E07B00", "vermelho": "#C62828",
         "cinza": "#757575", "azul_claro": "#5B9BD5", "roxo": "#6A1B9A"}

def brl(x, casas=2):
    """Formata um número como R$ 1.234,56."""
    s = f"{x:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return "R$ " + s

def pct(x, casas=2):
    return f"{x*100:,.{casas}f}%".replace(",", "X").replace(".", ",").replace("X", ".")

def eixo_reais(ax):
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, p: brl(v, 0)))

def sgs(codigo, inicio="01/01/2000", fim=None):
    """Baixa uma série do Sistema Gerenciador de Séries do Banco Central (SGS).
    Séries diárias são baixadas em blocos de 10 anos, o limite da API."""
    fim = fim or dt.date.today().strftime("%d/%m/%Y")
    d0 = dt.datetime.strptime(inicio, "%d/%m/%Y").date()
    d1 = dt.datetime.strptime(fim, "%d/%m/%Y").date()
    partes = []
    while d0 <= d1:
        d_fim = min(d1, d0.replace(year=d0.year + 9, month=12, day=31))
        url = (f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"
               f"?formato=json&dataInicial={d0:%d/%m/%Y}&dataFinal={d_fim:%d/%m/%Y}")
        for tentativa in range(4):
            try:
                r = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
                r.raise_for_status()
                dados = r.json()
                break
            except Exception:
                time.sleep(3 * (tentativa + 1))
        else:
            raise RuntimeError(f"Não consegui baixar a série {codigo}. Tente de novo em alguns minutos.")
        if dados:
            partes.append(pd.DataFrame(dados))
        d0 = d_fim + dt.timedelta(days=1)
    df = pd.concat(partes, ignore_index=True)
    df["data"] = pd.to_datetime(df["data"], dayfirst=True)
    df["valor"] = pd.to_numeric(df["valor"])
    return df.set_index("data")["valor"].sort_index()


## 1. As dívidas
O exemplo é o do CALCULE AGORA do capítulo 24. Troque pelos seus números.

In [ ]:
DIVIDAS = [   # MUDE AQUI: nome, saldo (R$), juros ao mês, parcela mínima (R$)
    ('Cartão parcelado', 2000, 0.09, 200),
    ('Crédito pessoal', 6000, 0.05, 400),
    ('Carnê de loja', 600, 0.03, 60),
]
PAGAMENTO_MENSAL = 900   # MUDE AQUI: quanto você consegue pagar por mês, no total

## 2. A simulação
Todo mês, cada dívida rende juros e recebe a parcela mínima. O que sobra vai para a dívida da vez: a de juros mais altos (avalanche) ou a de menor saldo (bola de neve). Quando uma acaba, a parcela dela passa para a próxima.

In [ ]:
def simular(dividas, pagamento, metodo):
    d = [dict(nome=n, saldo=s, juros=j, minimo=p) for n, s, j, p in dividas]
    mes = 0; juros_total = 0; quitacao = {}
    while any(x['saldo'] > 0.005 for x in d) and mes < 600:
        mes += 1
        for x in d:
            if x['saldo'] > 0:
                j = x['saldo'] * x['juros']; x['saldo'] += j; juros_total += j
        sobra = pagamento
        for x in d:
            if x['saldo'] > 0:
                p = min(x['minimo'], x['saldo']); x['saldo'] -= p; sobra -= p
        ativos = [x for x in d if x['saldo'] > 0.005]
        ordem = sorted(ativos, key=(lambda x: -x['juros']) if metodo == 'avalanche' else (lambda x: x['saldo']))
        for x in ordem:
            if sobra <= 0: break
            p = min(sobra, x['saldo']); x['saldo'] -= p; sobra -= p
        for x in d:
            if x['saldo'] <= 0.005 and x['nome'] not in quitacao:
                quitacao[x['nome']] = mes; x['saldo'] = 0
    return mes, juros_total, quitacao

for metodo in ('avalanche', 'bola de neve'):
    meses, juros, q = simular(DIVIDAS, PAGAMENTO_MENSAL, metodo)
    print(f'{metodo.capitalize()}: sai das dívidas em {meses} meses e paga {brl(juros)} de juros')
    print('   quitação:', ', '.join(f'{k} no mês {v}' for k, v in q.items()))

A avalanche costuma sair mais barata; a bola de neve dá vitórias rápidas, que ajudam quem precisa de motivação. O importante é escolher um método e não parar. Simule também na [calculadora de quitar dívidas](https://www.traderbrasil.com/blog/calculadora-quitar-dividas-bola-de-neve-avalanche.php).

---
**Hub do livro:** https://www.traderbrasil.com/livro-dinheiro/ · **Capítulo 24 no hub:** https://www.traderbrasil.com/livro-dinheiro/cap24/

Achou um erro? Escreva para contato@traderbrasil.com.